In [ ]:
def get_com(x, y, z, m):
    return [np.sum(m * axis) / np.sum(m) for axis in [x, y, z]]


def shrinking_sphere(x, y, z, m, r=100, rate=0.95, history=None, limit=1000):
    if rate >= 1: return

    count = 0
    
    guess = [np.median(axis)for axis in [x, y, z]]

    bound_x = x.copy()
    bound_y = y.copy()
    bound_z = z.copy()
    bound_m = m.copy()
    
    while True:
        if count > limit: 
            print('limit exceeded')
            return 
        
        if history is not None:
            history.append(guess)
        
        radii = np.sqrt( (bound_x - guess[0])**2 + (bound_y - guess[1])**2 + (bound_z - guess[2])**2 )
        
        idx = radii <= r

        if np.sum(idx) < 0.01 * len(x): return guess

        bound_x = bound_x[idx]
        bound_y = bound_y[idx]
        bound_z = bound_z[idx]
        bound_m = bound_m[idx]

        guess = get_com(bound_x, bound_y, bound_z, bound_m)

        r *= rate
        count += 1
        

    
    

In [12]:
def find_bounding_radii(x, y, z, m, fractions):
    m_tot = np.sum(m)
    com = shrinking_sphere(x, y, z, m)

    radii = np.sqrt( (x - com[0])**2 + (y - com[1])**2 + (z - com[2])**2 )

    m_fractions = mtot * fractions

    idx = np.argsort(radii)
    radii_sort = radii[idx]
    m_sort = m[idx]
    
    m_cumul = np.cumsum(m_sort)
    
    frac_mass_radii = radii_sort[np.searchsorted(m_cumul, m_fractions)]
    return frac_mass_radii
    



def find_mass_profile(x, y, z, m):
    com = shrinking_sphere(x, y, z, m)

    radii = np.sqrt( (x - com[0])**2 + (y - com[1])**2 + (z - com[2])**2 )

    idx = np.argsort(radii)
    radii_sort = radii[idx]
    m_sort = m[idx]
    
    return radii_sort, m_sort, np.cumsum(m_sort)

def find_mass_profiles(run_data):
    mass_profiles = {time : find_mass_profile(df['X'], df['Y'], df['Z'], df['M'])
                                                      for time,df in run_data.items()}
    return mass_profiles

def find_mass_profiles_experiment(runs_data):
    all_mass_profiles = {run : find_mass_profiles(data) for run, (data,_,_) in runs_data.items()}
    return all_mass_profiles


In [14]:
def plot_mass_profile(fig, ax, mass_profiles, 
                      xlbl=r'$r$ / $pc$', ylbl=r'$M(<r)$ / $M_\odot$', title='',
                      legend=False
                      ):
    for time, (r_sort, _, m_sums) in mass_profiles.items():
        ax.plot(r_sort, m_sums, label=f'{time:.1f}Myr')
        
    ax.set_xscale('log')
    #plt.yscale('log')
    
    if legend: ax.legend()
    ax.set_xlabel(xlbl)
    ax.set_ylabel(ylbl)
    ax.set_title(title)

def plot_mass_profiles(fig, axes, runs_mass_profiles, titlehead='', legend=False):
    for ax, (run, mass_profiles) in zip(axes.flat, runs_mass_profiles.items()):
        plot_mass_profile(fig, ax, mass_profiles, title=rf'{titlehead}{run}', legend=legend)
        